# T-Lab RecSys 2026: Causal GPT-2 Sequential Recommender System
### End-to-End Training on Kaggle with NVIDIA GPU (T4 / P100) & Mixed Precision (FP16)

This notebook runs the **Causal GPT-2 Next-Item Prediction** pipeline on the **Yambda** dataset (Yandex Music) initialized from scratch.

**Pre-flight Checklist on Kaggle:**
1. In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x 2** or **GPU P100**.
2. In **Settings -> Internet**, toggle **Internet ON** (needed for dependencies and Git).
3. Attach your Yambda dataset under `/kaggle/input/` (via "+ Add Input" in the top right).

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Clone Repository or Setup Workspace
If you pushed your repository to GitHub, clone it below. Alternatively, if you uploaded the `.py` files directly, make sure you are in the directory containing `train_base.py`.

In [ ]:
# If cloning from GitHub, uncomment and run:
# !git clone https://github.com/<your-username>/Music-Recommender-System.git
# %cd Music-Recommender-System

# Install project dependencies
!pip install --quiet -r requirements.txt
!ls -la

## 3. Optional: Weights & Biases (WandB) Authentication
You can securely store your `WANDB_API_KEY` in Kaggle under **Add-ons -> Secrets**.

In [ ]:
import os
USE_WANDB = False

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("[WandB] Successfully logged in via Kaggle Secrets!")
    USE_WANDB = True
except Exception as e:
    print("[WandB] No secret key found or offline mode. Running with local console logging.")

## 4. Dataset Discovery & Path Configuration
Checks for interaction logs in `/kaggle/input/`. If no dataset is mounted yet, it automatically enables `--synthetic` demo mode.

In [ ]:
import glob
from pathlib import Path

input_files = glob.glob("/kaggle/input/**/*", recursive=True)
parquet_files = [f for f in input_files if f.endswith(".parquet") or f.endswith(".csv")]

if parquet_files:
    data_path = parquet_files[0]
    synthetic_flag = ""
    print(f"[Data] Found dataset: {data_path}")
else:
    data_path = "/kaggle/working/data/synthetic_yambda.parquet"
    synthetic_flag = "--synthetic"
    print("[Data] No dataset found in /kaggle/input/. Using --synthetic generator for demonstration.")

## 5. Launch Training with Mixed Precision (FP16)
Executes `train_base.py` on the GPU with `--fp16`, `--split_mode global_temporal`, and evaluation on **HitRate@10** and **NDCG@10**.

In [ ]:
wandb_arg = "--use_wandb" if USE_WANDB else ""

!python train_base.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed \
    --output_dir /kaggle/working/checkpoints \
    --min_item_freq 5 \
    --max_length 64 \
    --n_embd 128 \
    --n_layer 3 \
    --n_head 4 \
    --batch_size 128 \
    --lr 5e-4 \
    --epochs 10 \
    --split_mode global_temporal \
    --fp16 \
    --num_workers 2 \
    {synthetic_flag} \
    {wandb_arg}

## 6. Model Verification & Top-K Recommendation Inference
Loads the best model checkpoint and tests next-item prediction on a user trajectory.

In [ ]:
import torch
from data_utils import ItemTokenizer
from model import build_gpt2_recsys_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = ItemTokenizer.load("/kaggle/working/data/processed/item_tokenizer.json")
checkpoint = torch.load("/kaggle/working/checkpoints/best_gpt2_recsys.pt", map_location=device)

cfg = checkpoint["model_config"]
model = build_gpt2_recsys_model(**cfg).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Best Model Checkpoint (Epoch {checkpoint['epoch']}): Val HR@10 = {checkpoint['val_hr10']:.4f}, Val NDCG@10 = {checkpoint['val_ndcg10']:.4f}")

# Demonstration: Sample next-item prediction
sample_input = torch.tensor([[1, 2, 3, 4]], dtype=torch.long, device=device)
scores, topk = model.predict_next_items(sample_input, top_k=5)
rec_items = tokenizer.inverse_transform(topk[0].cpu().numpy())
print(f"Input History: {sample_input[0].tolist()}")
print(f"Top-5 Recommended Next Tracks: {rec_items}")

## 7. Export Trained Artifacts
Compresses checkpoints and processed tokenizer mappings into a single `.zip` file for download from `/kaggle/working`.

In [ ]:
!zip -r /kaggle/working/recsys_checkpoints.zip /kaggle/working/checkpoints /kaggle/working/data/processed
print("Checkpoints successfully archived to: /kaggle/working/recsys_checkpoints.zip")